In [1]:
# --- Colab bootstrap -------------------------------------------------------
# No-op when you already have the thermo package alongside this notebook (the
# normal case: you cloned the repository and are running from code/chNN/).
# In Colab there is no repository, so fetch the package and the property data.
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# Illustration 6.7-1 &mdash; nitrogen discharged from a cylinder, with the Peng&ndash;Robinson equation of state

Nitrogen is withdrawn from a $0.15\ \mathrm{m^3}$ cylinder at 10 mol/min. The cylinder
starts at 100 bar and 170 K, is well insulated, and exchanges negligible heat with its
walls. How many moles remain after 50 minutes, and what are the temperature and pressure
then?

That is Illustration 6.5-1, and this notebook works it a **fifth** way. Section 6.5
solves it three times &mdash; as an ideal gas, from the nitrogen properties chart of
Fig. 3.3-3, and with the van der Waals equation &mdash; Illustration 6.6-2 solves it from
the corresponding-states charts, and Illustration 6.7-1 solves it with a *generalized*
cubic equation of state. Table 6.5-1 and the Summary table of Illustration 6.5-1 collect
all five answers side by side, which is the real point of the sequence: one problem, five
descriptions of the same fluid.

### The three equations

$$\text{mass:}\quad N(t) = N(t=0) + \dot N t, \qquad \dot N = -10\ \mathrm{mol/min}$$

$$\text{volume:}\quad \underline{V}(t) = \frac{V_{\mathrm{cyl}}}{N(t)}
\qquad\text{(Eq. e of Illustration 6.5-1)}$$

$$\text{entropy:}\quad \underline{S}(t=50\ \mathrm{min}) = \underline{S}(t=0)
\qquad\text{(Eq. c of Illustration 6.5-1)}$$

The third one carries the physics. Follow the *portion of the gas that never leaves* the
cylinder, as Illustration 4.5-2 does: it is a closed system, adiabatic because the
cylinder is insulated, and the expansion is taken to be reversible, so its entropy
balance reduces to $d\underline{S}/dt = 0$. The gas left behind cools because it does
work pushing the rest of the charge out.

### Why this is one equation in one unknown, not two in two

The mass balance alone fixes $\underline{V}(t=50)$ &mdash; no thermodynamics needed. With
$\underline{V}$ known, the equation of state makes $P$ a function of $T$ alone, so the
entropy condition is a **single** equation in the **single** unknown $T(t=50)$. That is
what makes the calculation short. The printed illustration describes a nested hand
iteration instead (guess $T$, iterate on $P$ until $\underline{V}$ comes out right, then
test the entropy); Section IV below runs it that way too, so you can watch the two
converge to the same answer.

### Inputs, and where each one comes from

| quantity | value | source |
|---|---|---|
| $T_c$ | 126.2 K | Table 6.6-1 |
| $P_c$ | 3.394 MPa | Table 6.6-1 |
| $\omega$ | 0.040 | Table 6.6-1 |
| $\kappa$ | computed from $\omega$ by Eq. 6.7-4 | *not* typed in as a rounded constant |
| $C_P^{*}$ | $27.2 + 4.2\times10^{-3}\,T$ J/(mol&nbsp;K) | the *Data* block of Illustration 6.5-1 |
| $R$ | 8.314462618 J/(mol&nbsp;K) | SI |

$C_P^{*}$ is **given by the problem**, and every one of the five rows of Table 6.5-1 uses
it. That is deliberate: holding the heat capacity fixed is what makes the table a
comparison of *equations of state* and of nothing else. Section VI asks what a more
accurate $C_P^{*}$ would do, and the answer is interesting enough to be worth a section
of its own.

In [2]:
import sys
sys.path.append("..")           # so `import thermo` finds code/thermo

import numpy as np
from scipy.optimize import brentq

from thermo import PengRobinson, VanDerWaals, TABLE_6_6_1
from thermo.peng_robinson import R

# The problem statement.
V_CYL = 0.15                    # m^3, the cylinder
NDOT = -10.0                    # mol/min, withdrawal rate (out of the cylinder)
T0, P0 = 170.0, 100e5           # K, Pa -- the initial state
T_ELAPSED = 50.0                # min

# Cp* = a + b T + c T^2 + d T^3, J/(mol K). The illustration gives two terms.
CP_STAR = (27.2, 4.2e-3, 0.0, 0.0)

# Nitrogen, built explicitly from the book's Table 6.6-1 rather than with
# `from_database`: the database is the Reid-Prausnitz-Poling table, whose row is
# close but not identical, and this illustration's printed numbers come from
# Table 6.6-1. See thermo/data.py.
pr = PengRobinson(**TABLE_6_6_1["nitrogen"], cp=CP_STAR)

print(f"R      = {R!r} J/(mol K)")
print(f"Tc     = {pr.Tc} K      Pc = {pr.Pc/1e6:.3f} MPa      omega = {pr.omega}")
print(f"kappa  = {pr.kappa:.6f}   (Eq. 6.7-4, from omega)")
print(f"b      = {pr.b:.6e} m3/mol         (Eq. 6.7-2)")
print(f"a(170) = {pr.a(T0):.6f} Pa m6/mol2   (Eqs. 6.7-1, 6.7-3)")
print(f"\nTr(t=0) = {T0/pr.Tc:.3f}   Pr(t=0) = {P0/pr.Pc:.3f}"
      f"   -- the same reduced state Illustration 6.6-2 reads off the charts")

R      = 8.31446261815324 J/(mol K)
Tc     = 126.2 K      Pc = 3.394 MPa      omega = 0.04
kappa  = 0.435899   (Eq. 6.7-4, from omega)
b      = 2.405256e-05 m3/mol         (Eq. 6.7-2)
a(170) = 0.128282 Pa m6/mol2   (Eqs. 6.7-1, 6.7-3)

Tr(t=0) = 1.347   Pr(t=0) = 2.946   -- the same reduced state Illustration 6.6-2 reads off the charts


## I. The initial state

Step 1 of the procedure in the text: the state is fully specified, so solve the cubic
form of the equation of state, Eq. 6.4-4,

$$Z^3 + (-1+B)Z^2 + (A - 3B^2 - 2B)Z + (-AB + B^2 + B^3) = 0,
\qquad A = \frac{a(T)P}{(RT)^2}, \quad B = \frac{bP}{RT}$$

take the vapor (largest) root, and get $\underline{V} = ZRT/P$ and then
$N(t=0) = V_{\mathrm{cyl}}/\underline{V}(t=0)$.

The entropy departure at this state comes from Eq. 6.4-30,

$$\bigl(\underline{S}-\underline{S}^{\mathrm{IG}}\bigr)_{T,P}
= R\ln(Z-B) + \frac{\mathrm{d}a/\mathrm{d}T}{2\sqrt{2}\,b}
\ln\!\left[\frac{Z+(1+\sqrt2)B}{Z+(1-\sqrt2)B}\right]$$

Note how far from ideal this state is: $Z = 0.68$ at 170 K and 100 bar, so an ideal-gas
calculation is wrong about the amount of nitrogen in the cylinder by about a third
&mdash; which is exactly what the first row of Table 6.5-1 shows, and it is the largest
single error anywhere in this problem.

**On the last digit of $N(t=0)$.** The book prints 1567.9 mol; this notebook gets
1567.8. Both are right. The printed value comes from dividing the cylinder volume by
$\underline{V}$ *as rounded for print*, $0.9567\times10^{-4}$; the notebook divides by the
unrounded $9.5675\times10^{-5}$. The difference is 0.1 mol in 1568, it propagates to
$\underline{V}(t=50)$ as $1.4047$ against a printed $1.4046\times10^{-4}$, and it changes
nothing at all in $T(t=50)$ or $P(t=50)$ &mdash; both come out 134.66 K and 40.56 bar
either way. Worth knowing before you conclude you have mistyped something.

In [3]:
roots = pr.physical_Z(T0, P0)
print(f"real roots of the cubic at 170 K, 100 bar: {np.round(roots, 6)}")

Z0 = max(roots)                 # vapor phase
V0 = Z0 * R * T0 / P0
N0 = V_CYL / V0
depS0 = pr.departure_S(T0, P0)

print(f"\nZ(t=0)   = {Z0:.4f}")
print(f"V(t=0)   = {V0:.4e} m3/mol")
print(f"N(t=0)   = {N0:.1f} mol")
print(f"(S - S_IG)(t=0) = {depS0:.2f} J/(mol K)")

# For comparison, the ideal-gas answer for the same cylinder:
N0_ideal = V_CYL / (R * T0 / P0)
print(f"\nan ideal gas would hold {N0_ideal:.1f} mol -- low by "
      f"{100*(1 - N0_ideal/N0):.0f}%, because Z = {Z0:.3f}, not 1")

real roots of the cubic at 170 K, 100 bar: [0.676886]

Z(t=0)   = 0.6769
V(t=0)   = 9.5675e-05 m3/mol
N(t=0)   = 1567.8 mol
(S - S_IG)(t=0) = -9.18 J/(mol K)

an ideal gas would hold 1061.2 mol -- low by 32%, because Z = 0.677, not 1


## II. Mass and volume balances

No thermodynamics here at all &mdash; this is Eqs. a and e of Illustration 6.5-1 and a
constant cylinder volume. It is worth doing on its own, because it is what turns two
unknowns into one.

In [4]:
N50 = N0 + NDOT * T_ELAPSED
V50 = V_CYL / N50

print(f"N(t=50 min) = {N0:.1f} + ({NDOT:.0f})({T_ELAPSED:.0f}) = {N50:.1f} mol")
print(f"V(t=50 min) = {V_CYL} m3 / {N50:.1f} mol = {V50:.4e} m3/mol")
print(f"\nthe gas expanded by a factor of {V50/V0:.3f} in molar volume")

N(t=50 min) = 1567.8 + (-10)(50) = 1067.8 mol
V(t=50 min) = 0.15 m3 / 1067.8 mol = 1.4047e-04 m3/mol

the gas expanded by a factor of 1.468 in molar volume


## III. The entropy condition, solved directly

Written out, $\underline{S}(t=50) = \underline{S}(t=0)$ is

$$0 = \underbrace{\int_{170}^{T}\frac{C_P^{*}}{T'}\,\mathrm{d}T'
- R\ln\frac{P}{P(t=0)}}_{\text{ideal-gas change}}
+ \underbrace{\bigl(\underline{S}-\underline{S}^{\mathrm{IG}}\bigr)_{T,P}
- \bigl(\underline{S}-\underline{S}^{\mathrm{IG}}\bigr)_{t=0}}_{\text{departures, from the EOS}}$$

which is Eq. b of the printed illustration. Everything on the right is known once $T$ is
guessed, because $P = P(\underline{V}(t=50), T)$ from Eq. 6.4-2.

**One implementation detail worth pausing on.** The departure is a function of state, and
we already know the state completely: $T$ and $\underline{V}$. Computing $P$ and then
calling a departure function that *re-solves* the cubic for $Z$ would be a round trip
through an equation with up to three roots, and at low enough temperature the root it
picks need not be the $\underline{V}$ we started from. So evaluate the departure from
$T$ and $\underline{V}$ directly, with $Z = P\underline{V}/RT$. Same equation, no root to
choose.

In [5]:
SQRT2 = np.sqrt(2.0)
cA, cB, cC, cD = pr.cp


def S_ideal_change(T, P):
    """Ideal-gas entropy change from (T0, P0) to (T, P), J/(mol K)."""
    return (cA * np.log(T / T0) + cB * (T - T0)
            + cC / 2 * (T**2 - T0**2) + cD / 3 * (T**3 - T0**3)
            - R * np.log(P / P0))


def departure_S_at_V(T, V):
    """Eq. 6.4-30, evaluated from T and V -- no cubic to re-solve, no root to pick."""
    P = pr.pressure(V, T)
    Z, B = P * V / (R * T), pr.b * P / (R * T)
    log_term = np.log((Z + (1 + SQRT2) * B) / (Z + (1 - SQRT2) * B))
    return R * np.log(Z - B) + pr.dadT(T) / (2 * SQRT2 * pr.b) * log_term


def entropy_residual(T):
    """S(t=50) - S(t=0), J/(mol K). Zero at the answer."""
    P = pr.pressure(V50, T)
    return S_ideal_change(T, P) + departure_S_at_V(T, V50) - depS0


# sanity check: the residual must change sign across the bracket
print(f"residual at 120 K: {entropy_residual(120.0):+.3f} J/(mol K)")
print(f"residual at 165 K: {entropy_residual(165.0):+.3f} J/(mol K)")

T50 = brentq(entropy_residual, 120.0, 165.0, xtol=1e-10)
P50 = pr.pressure(V50, T50)
depS50 = departure_S_at_V(T50, V50)

print(f"\nT(t=50 min) = {T50:.2f} K")
print(f"P(t=50 min) = {P50/1e5:.2f} bar")
print(f"(S - S_IG)(t=50) = {depS50:.2f} J/(mol K)")
print(f"\nresidual at the solution: {entropy_residual(T50):.2e} J/(mol K)")

# The terms of Eq. b, so you can see which one does the work.
print(f"\n  ideal-gas Cp* integral   {cA*np.log(T50/T0) + cB*(T50-T0):+7.3f}")
print(f"  -R ln(P/P0)              {-R*np.log(P50/P0):+7.3f}")
print(f"  departure at t = 50      {depS50:+7.3f}")
print(f"  departure at t = 0       {-depS0:+7.3f}")
print(f"  {'-'*33}\n  sum                      {entropy_residual(T50):+7.3f}")

residual at 120 K: -2.498 J/(mol K)
residual at 165 K: +4.390 J/(mol K)

T(t=50 min) = 134.66 K
P(t=50 min) = 40.56 bar
(S - S_IG)(t=50) = -10.19 J/(mol K)

residual at the solution: 3.55e-15 J/(mol K)

  ideal-gas Cp* integral    -6.488
  -R ln(P/P0)               +7.503
  departure at t = 50      -10.193
  departure at t = 0        +9.178
  ---------------------------------
  sum                       +0.000


## IV. The same answer by hand

The printed illustration describes a nested iteration, and it is worth running once
because it shows what the root finder in Section III is doing:

1. Guess $T(t=50)$. The ideal-gas result is a good first guess.
2. Iterate on $P$ until the equation of state returns the known
   $\underline{V}(t=50)$ at that guessed $T$.
3. Check Eq. b. If it is not satisfied, adjust $T$ and go back to step 2.

Step 2 is written below as a root find on $P$ &mdash; the same trial and error, done by
bisection instead of by pencil &mdash; so that nothing in this loop uses knowledge that
the hand calculation would not have. It has been given a deliberately poor starting
guess, and the table shows how the entropy residual shrinks.

In [6]:
def pressure_from_V(T, V_target):
    """Step 2: find P such that the EOS gives V_target at this T (vapor root)."""
    def resid(P):
        return max(pr.physical_Z(T, P)) * R * T / P - V_target
    return brentq(resid, 1e5, 300e5, xtol=1e-6)


def eq_b(T, P):
    """Step 3: the left-hand side of Eq. b. Zero when the guess is right."""
    return S_ideal_change(T, P) + departure_S_at_V(T, V50) - depS0


print(f"{'trial':>6}{'guess T (K)':>13}{'P (bar)':>10}{'Eq. b residual':>17}")
print("-" * 46)

# Two starting guesses, then adjust T by the secant rule -- which is all
# "adjust the guessed value of T and repeat" amounts to.
T_a, T_b = 129.6, 145.0             # the ideal-gas answer, and one deliberately high
P_a = pressure_from_V(T_a, V50)
P_b = pressure_from_V(T_b, V50)
r_a, r_b = eq_b(T_a, P_a), eq_b(T_b, P_b)
for k, (T, P, r) in enumerate([(T_a, P_a, r_a), (T_b, P_b, r_b)], start=1):
    print(f"{k:6d}{T:13.2f}{P/1e5:10.2f}{r:+17.4f}")

for k in range(3, 12):
    T_c = T_b - r_b * (T_b - T_a) / (r_b - r_a)
    P_c = pressure_from_V(T_c, V50)          # step 2, by trial and error on P
    r_c = eq_b(T_c, P_c)                     # step 3, the entropy test
    print(f"{k:6d}{T_c:13.2f}{P_c/1e5:10.2f}{r_c:+17.4f}")
    T_a, r_a, T_b, r_b = T_b, r_b, T_c, r_c
    if abs(r_c) < 1e-8:
        break

T_guess, P_guess = T_b, pressure_from_V(T_b, V50)
print(f"\nby hand:        T = {T_guess:.2f} K, P = {P_guess/1e5:.2f} bar")
print(f"by root finder:  T = {T50:.2f} K, P = {P50/1e5:.2f} bar")
assert abs(T_guess - T50) < 0.01, "the two routes disagree"
print("\nSame answer, as it must be -- the hand procedure and the simultaneous")
print("solve are the same two equations, solved in a different order.")

 trial  guess T (K)   P (bar)   Eq. b residual
----------------------------------------------
     1       129.60     35.98          -0.8288
     2       145.00     49.84          +1.6008
     3       134.85     40.74          +0.0318
     4       134.65     40.55          -0.0012
     5       134.66     40.56          +0.0000
     6       134.66     40.56          +0.0000

by hand:        T = 134.66 K, P = 40.56 bar
by root finder:  T = 134.66 K, P = 40.56 bar

Same answer, as it must be -- the hand procedure and the simultaneous
solve are the same two equations, solved in a different order.


## V. The five methods side by side

This is Table 6.5-1 and the Summary table of Illustration 6.5-1. Four of the five rows
are calculations and are recomputed here; the corresponding-states row of Illustration
6.6-2 is **read off charts by eye** and cannot be recomputed, so its printed values are
carried in for comparison and labeled.

The van der Waals parameters come from the critical properties through Eq. 6.6-4a, and
its entropy departure is the much simpler $R\ln[(\underline{V}-b)P/RT]$, since $a$ does
not depend on temperature.

Read the table as a sequence, not a scoreboard. The ideal gas is wrong about the *amount*
of nitrogen in the cylinder by a third, which is the largest error in the problem and has
nothing to do with entropy. Fig. 3.3-3 is real nitrogen data and is the closest thing here
to an answer. The two cubics bracket it within a couple of kelvin, and the
three-parameter cubic is the one you would carry to a fluid that has no chart.

In [7]:
def solve_case(eos, label):
    """Work the whole illustration with any of the cubic EOS in `thermo`."""
    V_0 = max(eos.physical_Z(T0, P0)) * R * T0 / P0
    N_0 = V_CYL / V_0
    V_50 = V_CYL / (N_0 + NDOT * T_ELAPSED)

    def dep_S(T, V):
        P = eos.pressure(V, T)
        Z, B = P * V / (R * T), eos.b * P / (R * T)
        out = R * np.log(Z - B)
        if eos.dadT(T):                       # zero for van der Waals
            out += (eos.dadT(T) / (2 * SQRT2 * eos.b)
                    * np.log((Z + (1 + SQRT2) * B) / (Z + (1 - SQRT2) * B)))
        return out

    d0 = dep_S(T0, V_0)
    T = brentq(lambda T: S_ideal_change(T, eos.pressure(V_50, T))
               + dep_S(T, V_50) - d0, 110.0, 169.0, xtol=1e-10)
    return label, V_0, N_0, V_50, T, eos.pressure(V_50, T) / 1e5


# ideal gas: V = RT/P and the departures vanish
V0_ig = R * T0 / P0
N0_ig = V_CYL / V0_ig
V50_ig = V_CYL / (N0_ig + NDOT * T_ELAPSED)
T_ig = brentq(lambda T: S_ideal_change(T, R * T / V50_ig), 110.0, 169.0, xtol=1e-10)

rows = [("ideal gas", V0_ig, N0_ig, V50_ig, T_ig, R * T_ig / V50_ig / 1e5),
        ("Fig. 3.3-3 (chart, by eye)", 9.80e-5, 1529.8, 1.457e-4, 133.0, 39.0),
        solve_case(VanDerWaals(Tc=pr.Tc, Pc=pr.Pc, name="nitrogen", cp=CP_STAR),
                   "van der Waals"),
        ("corresponding states (chart, by eye)", 1.047e-4, 1432.7, 1.608e-4, 136.0, 41.0),
        solve_case(pr, "Peng-Robinson")]

hdr = f"{'method':<38}{'N(t=0)':>9}{'V(t=50)':>12}{'T(t=50)':>10}{'P(t=50)':>10}"
print(hdr)
print(f"{'':38}{'mol':>9}{'m3/mol':>12}{'K':>10}{'bar':>10}")
print("-" * len(hdr))
for label, _V0, N_0, V_50, T, P in rows:
    print(f"{label:<38}{N_0:9.1f}{V_50:12.4e}{T:10.1f}{P:10.1f}")

print("\nspread in T(t=50) across the two cubics and the chart: "
      f"{max(r[4] for r in rows[1:]) - min(r[4] for r in rows[1:]):.1f} K")

method                                   N(t=0)     V(t=50)   T(t=50)   P(t=50)
                                            mol      m3/mol         K       bar
-------------------------------------------------------------------------------
ideal gas                                1061.2  2.6727e-04     129.6      40.3
Fig. 3.3-3 (chart, by eye)               1529.8  1.4570e-04     133.0      39.0
van der Waals                            1589.8  1.3764e-04     133.1      39.5
corresponding states (chart, by eye)     1432.7  1.6080e-04     136.0      41.0
Peng-Robinson                            1567.8  1.4047e-04     134.7      40.6

spread in T(t=50) across the two cubics and the chart: 3.0 K


## VI. How much does the ideal-gas heat capacity matter?

The problem hands you $C_P^{*} = 27.2 + 4.2\times10^{-3}\,T$ J/(mol&nbsp;K), a two-term
fit. Compare it with the nitrogen rows of Appendix A.II at the temperatures this problem
actually visits, 135&ndash;170 K:

- the printed A.II row for combustion gases is valid **273&ndash;1800 K**, so using it
  here is an extrapolation of nearly 140 K;
- the A.II **cryogenic** row (100&ndash;700 K) is the one fitted for this range.

The two-term fit is about 1.2 J/(mol&nbsp;K) &mdash; 4% &mdash; below the true
$C_P^{*}$ of nitrogen at 170 K. That is worth something, and the cell below says how
much. The point is not that one answer is right and the other wrong: with
$\underline{V}(t=50)$ pinned by the mass balance, the *only* thing $C_P^{*}$ can move is
where the isentrope lands, and it moves it by an amount comparable to the difference
between the two equations of state in Section V. A property calculation is only as good
as the *weakest* of its inputs, and here that is not the equation of state.

This is the same lesson as `ch3/Heat_capacity_range_of_validity.ipynb`: a correlation's
range of validity is part of its data, not a footnote to it.

In [8]:
from thermo import APPENDIX_A2_CP, APPENDIX_A2_CP_CRYO


def solve_with_cp(cp):
    """Rework the illustration with a different Cp*, everything else unchanged."""
    a, b, c, d = cp

    def dS_ideal(T, P):
        return (a * np.log(T / T0) + b * (T - T0) + c / 2 * (T**2 - T0**2)
                + d / 3 * (T**3 - T0**3) - R * np.log(P / P0))

    T = brentq(lambda T: dS_ideal(T, pr.pressure(V50, T))
               + departure_S_at_V(T, V50) - depS0, 110.0, 169.0, xtol=1e-10)
    return T, pr.pressure(V50, T) / 1e5


def cp_at(cp, T):
    a, b, c, d = cp
    return a + b * T + c * T**2 + d * T**3


cases = [("given: 27.2 + 4.2e-3 T (2 terms)", CP_STAR),
         ("Appendix A.II, 273-1800 K row", APPENDIX_A2_CP["nitrogen"]),
         ("Appendix A.II, cryogenic 100-700 K row", APPENDIX_A2_CP_CRYO["nitrogen"])]

hdr = (f"{'ideal-gas heat capacity':<40}{'Cp*(170 K)':>12}{'T(t=50)':>10}"
       f"{'P(t=50)':>10}")
print(hdr)
print(f"{'':40}{'J/(mol K)':>12}{'K':>10}{'bar':>10}")
print("-" * len(hdr))
for label, cp in cases:
    T, P = solve_with_cp(cp)
    print(f"{label:<40}{cp_at(cp, 170.0):12.2f}{T:10.2f}{P:10.2f}")

T_cryo, P_cryo = solve_with_cp(APPENDIX_A2_CP_CRYO["nitrogen"])
print(f"\nCp* alone moves the answer by {T_cryo - T50:+.2f} K and "
       f"{P_cryo - P50/1e5:+.2f} bar.")
print("For scale, van der Waals and Peng-Robinson differ by "
      f"{T50 - rows[2][4]:.2f} K on the same Cp*.")

ideal-gas heat capacity                   Cp*(170 K)   T(t=50)   P(t=50)
                                           J/(mol K)         K       bar
------------------------------------------------------------------------
given: 27.2 + 4.2e-3 T (2 terms)               27.91    134.66     40.56
Appendix A.II, 273-1800 K row                  28.84    136.02     41.79
Appendix A.II, cryogenic 100-700 K row         29.10    136.40     42.13

Cp* alone moves the answer by +1.75 K and +1.57 bar.
For scale, van der Waals and Peng-Robinson differ by 1.59 K on the same Cp*.


## What to take away

1. **Fix the volume first.** The mass balance and a constant cylinder volume give
   $\underline{V}(t=50)$ with no thermodynamics at all, and that collapses a
   two-unknown problem to one equation in $T$.
2. **A real-fluid property is an ideal-gas part plus a departure.** Every equation in
   this notebook has that shape; only the departure changes when the equation of state
   changes.
3. **Evaluate departures at the state you know.** Here that is $(T, \underline{V})$.
   Going out to $P$ and back through the cubic invites the wrong root.
4. **The weakest input sets the accuracy.** Section VI moves the answer as far with the
   heat capacity as Section V does by replacing van der Waals with Peng&ndash;Robinson.

The cell below writes the printed results to `output/` so they can be checked against
the book without rereading the notebook.

In [9]:
from pathlib import Path

Path("output").mkdir(exist_ok=True)
lines = [
    "Illustration 6.7-1 -- nitrogen withdrawn from a 0.15 m3 cylinder",
    "Peng-Robinson EOS, Table 6.6-1 constants, Cp* = 27.2 + 4.2e-3 T J/(mol K)",
    "",
    f"  Tc = {pr.Tc} K   Pc = {pr.Pc/1e6:.3f} MPa   omega = {pr.omega}"
    f"   kappa = {pr.kappa:.4f}",
    f"  b  = {pr.b:.4e} m3/mol   a(170 K) = {pr.a(T0):.4f} Pa m6/mol2",
    "",
    "  t = 0 (170 K, 100 bar)",
    f"    Z              = {Z0:.4f}",
    f"    V              = {V0:.4e} m3/mol",
    f"    N              = {N0:.1f} mol",
    f"    S - S_IG       = {depS0:.2f} J/(mol K)",
    "",
    f"  t = {T_ELAPSED:.0f} min",
    f"    N              = {N50:.1f} mol",
    f"    V              = {V50:.4e} m3/mol",
    f"    T              = {T50:.2f} K",
    f"    P              = {P50/1e5:.2f} bar",
    f"    S - S_IG       = {depS50:.2f} J/(mol K)",
    "",
    "  the five methods of Table 6.5-1 (chart rows read by eye, not recomputed)",
    f"    {'method':<38}{'N(t=0)':>9}{'V(t=50)':>12}{'T':>8}{'P':>8}",
]
for label, _V0, N_0, V_50, T, P in rows:
    lines.append(f"    {label:<38}{N_0:9.1f}{V_50:12.4e}{T:8.1f}{P:8.1f}")

text = "\n".join(lines) + "\n"
Path("output/Illustration_6.7-1.txt").write_text(text)
print(text)

Illustration 6.7-1 -- nitrogen withdrawn from a 0.15 m3 cylinder
Peng-Robinson EOS, Table 6.6-1 constants, Cp* = 27.2 + 4.2e-3 T J/(mol K)

  Tc = 126.2 K   Pc = 3.394 MPa   omega = 0.04   kappa = 0.4359
  b  = 2.4053e-05 m3/mol   a(170 K) = 0.1283 Pa m6/mol2

  t = 0 (170 K, 100 bar)
    Z              = 0.6769
    V              = 9.5675e-05 m3/mol
    N              = 1567.8 mol
    S - S_IG       = -9.18 J/(mol K)

  t = 50 min
    N              = 1067.8 mol
    V              = 1.4047e-04 m3/mol
    T              = 134.66 K
    P              = 40.56 bar
    S - S_IG       = -10.19 J/(mol K)

  the five methods of Table 6.5-1 (chart rows read by eye, not recomputed)
    method                                   N(t=0)     V(t=50)       T       P
    ideal gas                                1061.2  2.6727e-04   129.6    40.3
    Fig. 3.3-3 (chart, by eye)               1529.8  1.4570e-04   133.0    39.0
    van der Waals                            1589.8  1.3764e-04   133.1    39.